# Bonus +10% — t-SNE / UMAP de conversaciones reales del agente Sándwich Qbano

Notebook anexo del Taller 3 (Ruta Transversal A — opcional avanzado).

Lo que hacemos acá:

1. Levantamos del Postgres todas las respuestas del agente (`role=assistant`, `route IS NOT NULL`).
2. Las vectorizamos con el mismo modelo de embeddings que usa el RAG (`paraphrase-multilingual-MiniLM-L12-v2`, 384d).
3. Proyectamos a 2D con **t-SNE** (preserva estructura local) y **UMAP** (preserva estructura global).
4. Calculamos el silhouette score por ruta para tener una métrica defendible.
5. Interpretamos los clústeres y discutimos qué se podría hacer con más datos.

Todo el código vive en `scripts/run_tsne_analysis.py` para que sea ejecutable también desde la línea de comandos. Este notebook es la versión "para sustentar en clase".

## 1. Setup — imports + conexión a Postgres

In [ ]:
import sys
from pathlib import Path
from collections import Counter

ROOT = Path.cwd().resolve()
while ROOT.name != 'proyecto' and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import psycopg
import plotly.express as px
from dotenv import load_dotenv
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import LabelEncoder
import umap

load_dotenv(dotenv_path=ROOT / '.env', override=True)

from src.checkpointer import get_db_url
from src.vector_store import get_embeddings

print('Setup OK — corpus path:', ROOT)

## 2. Levantamos los turnos del agente desde Postgres

Solo respuestas del asistente con `route` asignada. Los turnos del usuario no llevan `route` (la decisión de routing es del agente, no del usuario), por eso no los vectorizamos.

In [ ]:
with psycopg.connect(get_db_url()) as conn:
    rows = conn.execute("""
        SELECT content, route, thread_id
        FROM conversation_messages
        WHERE role = 'assistant' AND route IS NOT NULL AND length(trim(content)) > 0
        ORDER BY id
    """).fetchall()

texts   = [r[0] for r in rows]
routes  = [r[1] for r in rows]
threads = [r[2] for r in rows]

print(f'Total turnos: {len(texts)}')
print(f'Threads únicos: {len(set(threads))}')
print('Distribución por ruta:')
for route, count in sorted(Counter(routes).items(), key=lambda kv: -kv[1]):
    print(f'  {route:40s} {count:3d}  ({count/len(routes):.1%})')

## 3. Vectorizamos con el mismo modelo que usa el RAG

Esto es importante: si usáramos otro embeddings model, los clústeres podrían reflejar el sesgo del modelo, no las decisiones del agente. Manteniendo `paraphrase-multilingual-MiniLM-L12-v2` aseguramos que el análisis es coherente con lo que el agente ve en producción.

In [ ]:
embeddings = get_embeddings()
X = np.asarray(embeddings.embed_documents(texts))
print(f'Matriz de embeddings: {X.shape}  (turnos x dimensiones)')

## 4. Métrica de separabilidad — silhouette en el espacio original (384d)

Lo calculamos antes de reducir dimensionalidad para tener una medida que no dependa de t-SNE / UMAP. Usamos distancia coseno porque es la métrica natural para embeddings semánticos.

In [ ]:
y = LabelEncoder().fit_transform(routes)
sil = silhouette_score(X, y, metric='cosine')
print(f'Silhouette score (cosine, 384d original) = {sil:.4f}')
print('Interpretación:')
if sil > 0.5:
    print('  -> Separabilidad muy alta (clústeres bien definidos, pocas mezclas).')
elif sil > 0.3:
    print('  -> Separabilidad clara (clústeres distinguibles aunque con solapamientos).')
elif sil > 0.1:
    print('  -> Separabilidad moderada (los clústeres existen pero comparten frontera).')
else:
    print('  -> Separabilidad baja (los grupos se mezclan en el espacio embedding).')

## 5. t-SNE — proyección 2D que preserva estructura local

In [ ]:
perplexity = min(12, max(5, len(X) // 4))
tsne_coords = TSNE(
    n_components=2,
    perplexity=perplexity,
    metric='cosine',
    init='pca',
    learning_rate='auto',
    max_iter=1500,
    random_state=42,
).fit_transform(X)

fig_tsne = px.scatter(
    x=tsne_coords[:, 0],
    y=tsne_coords[:, 1],
    color=routes,
    hover_data={
        'preview': [t[:120].replace('\n', ' ') + ('…' if len(t) > 120 else '') for t in texts],
        'thread': threads,
    },
    title=f't-SNE 2D — {len(texts)} respuestas del agente coloreadas por ruta',
    labels={'x': 't-SNE 1', 'y': 't-SNE 2', 'color': 'Ruta'},
    opacity=0.8,
)
fig_tsne.update_traces(marker=dict(size=11, line=dict(width=0.4, color='white')))
fig_tsne.update_layout(template='plotly_white', width=1100, height=650)
fig_tsne.show()

## 6. UMAP — proyección 2D que preserva estructura global

Mostrar ambas técnicas es deliberado: si los clústeres se ven en las dos, la separación es robusta y no un artefacto de un algoritmo específico.

In [ ]:
umap_coords = umap.UMAP(
    n_components=2,
    n_neighbors=min(15, len(X) - 1),
    min_dist=0.1,
    metric='cosine',
    random_state=42,
).fit_transform(X)

fig_umap = px.scatter(
    x=umap_coords[:, 0],
    y=umap_coords[:, 1],
    color=routes,
    hover_data={
        'preview': [t[:120].replace('\n', ' ') + ('…' if len(t) > 120 else '') for t in texts],
        'thread': threads,
    },
    title=f'UMAP 2D — {len(texts)} respuestas del agente coloreadas por ruta',
    labels={'x': 'UMAP 1', 'y': 'UMAP 2', 'color': 'Ruta'},
    opacity=0.8,
)
fig_umap.update_traces(marker=dict(size=11, line=dict(width=0.4, color='white')))
fig_umap.update_layout(template='plotly_white', width=1100, height=650)
fig_umap.show()

## 7. Interpretación de clústeres

Lo que vemos en los gráficos:

- **Clúster grande, denso, bien separado — `consultar_datos_contacto`.** Estas respuestas siguen un molde fijo ("Encontré esta información estructurada: …") porque vienen de la tool determinística que lee del JSON. El embedding multilingüe captura ese molde y los agrupa con muy poca dispersión.

- **Clúster medio con sub-núcleos — `buscar_catalogo_productos`.** Aparece más disperso porque mezcla respuestas determinísticas (rankings de precios) con respuestas generadas por el LLM sobre RAG. Se notan dos sub-núcleos: uno de respuestas tabuladas, otro de respuestas sintetizadas por el modelo.

- **Puntos satélite — `memory`, `conversation`, `solicitar_supervisor_humano`.** Estos aparecen como pequeños grupos aislados en los bordes del gráfico. Indica que el agente *sí* responde distinto cuando la ruta es conversacional o de memoria personal, en lugar de caer siempre en plantillas RAG.

- **Solapamiento `consultar_informacion_corporativa` ↔ `buscar_catalogo_productos`.** Algunos puntos quedan en zona fronteriza. Esto refleja una realidad del agente: preguntas como "¿en qué ciudades están?" pueden caer en uno u otro dependiendo del fraseo. El embedding captura correctamente esa ambigüedad.

## Qué se podría hacer con más datos

El corpus actual son 158 respuestas: pruebas de desarrollo + la demo real de WhatsApp del 4 de junio. Si esto saliera a clientes durante un mes:

1. Aparecería un clúster propio para **conversaciones fallidas** (turnos donde el agente cayó al fallback cortés porque el LLM falló).
2. Picos en el clúster de `solicitar_supervisor_humano` se podrían correlacionar con días específicos = señal operativa accionable.
3. Sub-clústeres densos dentro de `consultar_informacion_corporativa` indicarían preguntas frecuentes mal resueltas que ameritan entrar al JSON estructurado para acelerar respuesta.

## Archivos también generados por `scripts/run_tsne_analysis.py`

| Archivo | Para qué |
|---------|---------|
| `results/tsne_2d_static.png` | Imagen embebida en el informe PDF |
| `results/tsne_2d_interactive.html` | Gráfico para abrir en navegador con hover |
| `results/umap_2d_interactive.html` | Versión UMAP del mismo análisis |
| `results/tsne_analysis.md` | Resumen escrito (este notebook es la versión interactiva) |